# Spark Setup

In [ ]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.3.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0 pyspark-shell'

from pathlib import Path
from pymongo import MongoClient, UpdateOne
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

HOST_IP = "192.168.64.1"
MONGO_URI = "mongodb://localhost:27017/"
MONGO_DB = "fit3182_a2"  # matches mongo_setup.py

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('FIT3182-A2')
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)

print("Debug: SparkSession has been created successfully.")


## Accepting Streams

In [ ]:
event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("batch_id", IntegerType()),
    StructField("car_plate", StringType()),
    StructField("camera_id", IntegerType()),
    StructField("timestamp", StringType()),
    StructField("speed_reading", DoubleType())
])

def read_camera_stream(topic, producer):
    """Read a Kafka topic and return a watermarked streaming DataFrame.

    Args:
        topic: Kafka topic name to subscribe to.
        producer: Source label string added as a 'source' column for traceability.

    Returns:
        Watermarked streaming DataFrame with parsed event fields.
    """
    return (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", f"{HOST_IP}:9092")
        .option("subscribe", topic)
        .option("startingOffsets", "latest")
        .option("failOnDataLoss", "false")
        .load()
        # Kafka value arrives as bytes — cast to string before JSON parsing
        .selectExpr("CAST(value AS STRING) as json_value")
        .select(from_json(col("json_value"), event_schema).alias("data"))
        .select("data.*")
        # Convert string timestamp to proper Spark TimestampType
        .withColumn("event_time", to_timestamp(col("timestamp")))
        # Tag with originating producer for traceability and debugging
        .withColumn("source", lit(producer))
        # Watermark: tolerate events arriving up to 10 minutes late.
        # Required on both sides of a stream-stream join for state eviction.
        .withWatermark("event_time", "10 minutes")
    )

camera_stream_a = read_camera_stream("camera-events-A", "1")
camera_stream_b = read_camera_stream("camera-events-B", "2")
camera_stream_c = read_camera_stream("camera-events-C", "3")

print("Debug: Kafka streams have been created for all three cameras.")


## Static Camera Enrichment

Join each stream against the static camera CSV to attach `position` and `speed_limit` to every event.

In [ ]:
camera_df = (
    spark.read.csv(f"{Path('..')}/data/camera.csv", header=True, inferSchema=True)
    .select("camera_id", "position", "speed_limit")
)

camera_df.show()
print(f"Debug: Camera loaded: {camera_df.count()} cameras.")

def join_stream_with_camera(stream):
    """Enrich a camera event stream with static camera metadata.

    Args:
        stream: Streaming DataFrame with a camera_id column.

    Returns:
        Streaming DataFrame with position and speed_limit columns added.
    """
    return stream.join(camera_df, on="camera_id", how="inner")

joined_stream_a = join_stream_with_camera(camera_stream_a)
joined_stream_b = join_stream_with_camera(camera_stream_b)
joined_stream_c = join_stream_with_camera(camera_stream_c)

print("Debug: Streams have been joined with camera data.")


## Task 2.1.4 — Instantaneous Violation Detection

In [ ]:
def get_instant_violations(stream):
    """Detect instantaneous speed violations from a single enriched stream.

    A violation occurs when speed_reading exceeds the camera's speed_limit.

    Args:
        stream: Enriched streaming DataFrame with speed_reading and speed_limit columns.

    Returns:
        Filtered streaming DataFrame of violating rows only.
    """
    return (
        stream
        .filter(col("speed_reading") > col("speed_limit"))
        .withColumn("violation_type", lit("instantaneous"))
        .withColumn("date", to_date(col("event_time")))  # matches mongo_setup.py index field name
        .select(
            "car_plate",
            "date",
            "violation_type",
            "camera_id",
            col("speed_reading").alias("speed_recorded"),
            "speed_limit",
            col("event_time").cast("string").alias("event_time"),
            "source"
        )
    )

camera_a_instant_violations = get_instant_violations(joined_stream_a)
camera_b_instant_violations = get_instant_violations(joined_stream_b)
camera_c_instant_violations = get_instant_violations(joined_stream_c)

all_instant_violations = (
    camera_a_instant_violations
    .union(camera_b_instant_violations)
    .union(camera_c_instant_violations)
)

print("Debug: Instantaneous violation detection defined.")


## Task 2.1.2 — Stream-Stream Segment Join (Average Speed)

Two separate joins are defined — one per adjacent camera pair — rather than aliasing a union against itself.
This avoids the Spark self-join ambiguity issue where both sides share the same logical plan.

| Join | Entry stream | Exit stream | Segment |
|---|---|---|---|
| `segment_ab` | Producer A (camera 1) | Producer B (camera 2) | 1 → 2 |
| `segment_bc` | Producer B (camera 2) | Producer C (camera 3) | 2 → 3 |

The time bound (`interval 10 minutes`) is required so Spark can safely evict expired join state.
Without it, Spark buffers all events indefinitely and may never emit results.


In [ ]:
def build_segment_join(entry_stream, exit_stream):
    """Join two enriched streams to detect vehicles crossing a camera segment.

    Matches entry and exit events for the same vehicle where the exit occurs
    after the entry and within a 10-minute window (sufficient for a 1 km segment).

    Args:
        entry_stream: Enriched stream for the upstream (entry) camera.
        exit_stream: Enriched stream for the downstream (exit) camera.

    Returns:
        DataFrame of matched (entry, exit) pairs with segment travel details.
    """
    return (
        entry_stream.alias("entry")
        .join(
            exit_stream.alias("exit"),
            expr("""
                entry.car_plate = exit.car_plate
                AND exit.event_time > entry.event_time
                AND exit.event_time <= entry.event_time + interval 10 minutes
            """),
            "inner"
        )
        .select(
            col("entry.car_plate").alias("car_plate"),
            col("entry.camera_id").alias("segment_start_camera"),
            col("exit.camera_id").alias("camera_id"),          # end camera (used for speed limit)
            col("entry.event_time").alias("entry_time"),
            col("exit.event_time").alias("exit_time"),
            col("entry.position").alias("entry_position"),
            col("exit.position").alias("exit_position"),
            col("exit.speed_limit").alias("speed_limit"),      # rule: use END camera limit
            col("exit.source").alias("source"),
        )
    )

# A→B: camera 1 to camera 2
segment_ab = build_segment_join(joined_stream_a, joined_stream_b)

# B→C: camera 2 to camera 3
segment_bc = build_segment_join(joined_stream_b, joined_stream_c)

print("Debug: Segment joins defined for A→B and B→C.")


## Task 2.1.4 — Average Speed Calculation

In [ ]:
def get_average_violations(segment_df):
    """Compute average speed across a segment and flag violations.

    Average speed (km/h) = distance (km) / travel time (hours).
    A violation is raised when average speed exceeds the end-camera speed limit.

    Args:
        segment_df: DataFrame from build_segment_join containing entry/exit times and positions.

    Returns:
        DataFrame of average-speed violating rows only.
    """
    return (
        segment_df
        .withColumn(
            "distance_km",
            abs(col("exit_position") - col("entry_position"))
        )
        .withColumn(
            "travel_time_hours",
            (unix_timestamp(col("exit_time")) - unix_timestamp(col("entry_time"))) / 3600.0
        )
        .withColumn(
            "speed_recorded",
            col("distance_km") / col("travel_time_hours")
        )
        # Only flag if average speed exceeds the end camera's limit
        .filter(col("speed_recorded") > col("speed_limit"))
        .withColumn("violation_type", lit("average"))
        .withColumn("date", to_date(col("exit_time")))  # matches mongo_setup.py index field name
        .select(
            "car_plate",
            "date",
            "violation_type",
            "camera_id",
            "speed_recorded",
            "speed_limit",
            col("exit_time").cast("string").alias("event_time"),
            "segment_start_camera",
            "distance_km",
            "source",
        )
    )

avg_violations_ab = get_average_violations(segment_ab)
avg_violations_bc = get_average_violations(segment_bc)

all_avg_violations = avg_violations_ab.union(avg_violations_bc)

print("Debug: Average speed violation detection defined.")


## Task 2.1.3 — MongoDB Sink

Violations are written via `foreachBatch` using pymongo `bulk_write` with `UpdateOne` upserts.

**Daily merging (Task 2.1.4):** Multiple violations for the same `(car_plate, date, violation_type, camera_id)` 
are merged into one document. Each new incident is appended to an `incidents` array via `$push`, 
rather than creating a duplicate document. This matches the index defined in `mongo_setup.py` on `(car_plate, date)`.

**Idempotency:** Upserts are safe to retry — re-writing the same event just pushes a duplicate 
into the `incidents` array, which is preferable to crashing the stream.


In [ ]:
def write_violations_to_mongo(batch_df, batch_id):
    """foreachBatch sink: upsert violation records into MongoDB fit3182_a2.violations.

    Uses bulk_write for efficiency. Each violation is upserted keyed on
    (car_plate, date, violation_type, camera_id) to enforce daily merging —
    matching the compound index created in mongo_setup.py.

    Args:
        batch_df: Spark DataFrame for the current micro-batch.
        batch_id: Spark-assigned integer batch identifier (used for logging).
    """
    rows = batch_df.collect()

    if not rows:
        print(f"[Batch {batch_id}] No violations to write.")
        return

    operations = []
    for row in rows:
        # Compound upsert key — one document per car per day per camera per violation type
        filter_key = {
            "car_plate":      row["car_plate"],
            "date":           str(row["date"]),       # field name matches mongo_setup.py index
            "violation_type": row["violation_type"],
            "camera_id":      row["camera_id"],
        }

        # $setOnInsert only writes these fields when creating a new document
        # $push appends each new speed reading to the incidents array
        update_doc = {
            "$setOnInsert": {
                "speed_limit": row["speed_limit"],
                "source":      row["source"],
            },
            "$push": {
                "incidents": {
                    "speed_recorded": row["speed_recorded"],
                    "event_time":     row["event_time"],
                }
            }
        }

        # Add segment metadata for average violations only
        if row["violation_type"] == "average":
            update_doc["$setOnInsert"]["segment_start_camera"] = row["segment_start_camera"]
            update_doc["$setOnInsert"]["distance_km"] = row["distance_km"]

        operations.append(UpdateOne(filter_key, update_doc, upsert=True))

    # Open a fresh MongoClient per batch — pymongo is not serialisable across batches
    client = MongoClient(MONGO_URI)
    try:
        collection = client[MONGO_DB]["violations"]
        result = collection.bulk_write(operations, ordered=False)
        print(
            f"[Batch {batch_id}] Wrote {len(operations)} violation(s) — "
            f"upserted: {result.upserted_count}, modified: {result.modified_count}"
        )
    except Exception as exc:
        # Log and continue — do not crash the stream on a transient write error
        print(f"[Batch {batch_id}] MongoDB write error: {exc}")
    finally:
        client.close()


print("Debug: MongoDB sink function defined.")


## Logging Helper

In [ ]:
def log_batch(name):
    """Return a foreachBatch logger function that prints batch metadata and rows.

    Args:
        name: Label printed in the batch header for identification.

    Returns:
        A foreachBatch-compatible function (batch_df, batch_id) -> None.
    """
    def logger(batch_df, batch_id):
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        row_count = batch_df.count()
        print("\n" + "=" * 70)
        print(f"[{now}] {name}")
        print(f"Spark Batch ID: {batch_id} | Rows received: {row_count}")
        print("=" * 70)
        batch_df.show(truncate=False)
    return logger


print("Debug: Logging helper defined.")


## Start Streaming Queries

In [ ]:
import os

os.makedirs("checkpoints/instant_mongo",  exist_ok=True)
os.makedirs("checkpoints/average_mongo",  exist_ok=True)
os.makedirs("checkpoints/instant_debug",  exist_ok=True)
os.makedirs("checkpoints/average_debug",  exist_ok=True)

# --- Debug queries (console) ---
instant_debug_query = (
    all_instant_violations.writeStream
    .outputMode("append")
    .foreachBatch(log_batch("Instantaneous Violations"))
    .option("checkpointLocation", "checkpoints/instant_debug")
    .start()
)

average_debug_query = (
    all_avg_violations.writeStream
    .outputMode("append")
    .foreachBatch(log_batch("Average Speed Violations"))
    .option("checkpointLocation", "checkpoints/average_debug")
    .start()
)

# --- MongoDB sink queries ---
instant_mongo_query = (
    all_instant_violations.writeStream
    .outputMode("append")
    .foreachBatch(write_violations_to_mongo)
    .option("checkpointLocation", "checkpoints/instant_mongo")
    .start()
)

average_mongo_query = (
    all_avg_violations.writeStream
    .outputMode("append")
    .foreachBatch(write_violations_to_mongo)
    .option("checkpointLocation", "checkpoints/average_mongo")
    .start()
)

print("Debug: All streaming queries started.")
spark.streams.awaitAnyTermination()


## Stop All Queries

In [ ]:
for q in spark.streams.active:
    q.stop()
    print(f"Stopped: {q.name or q.id}")
print("All streaming queries stopped.")


In [ ]:
# Instant
{
"car_plate":"CIY 810",
"batch_id":3,
"violation_date": "2024-01-01",
"violation_type": "instantaneous",
"camera_id":1,
"speed_recorded":132.1,
"speed_limit":110,
"event_time":"2024-01-01 08:13:08",
"source":"1"
}

In [ ]:
# Average
{"car_plate":"YXA 7534",
"violation_date":"2024-01-01",
"violation_type":"average",
"end_camera_id":2,
"average_speed":149.238751481931,
"speed_limit":110,
"event_time":"2024-01-01 08:20:09.042838",
"start_camera_id":1,
"distance_km":0.9967008721171039,
"source":"2",
"entry_time":"2024-01-01T08:19:45.000Z",
"exit_time":"2024-01-01T08:20:09.042Z",
"entry_batch_id":4,
"exit_batch_id":30
}